# M19b — Validation-safe-region controls

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Compare the validation-derived region with same-width contiguous control intervals and examine movement inside versus outside that region.

**Provenance.** The original M19b binary is unavailable. Historical reported values are preserved as archival references. The executable part below is a fresh protocol replication using a repository-defined seed and the preserved later synthetic task generator.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

FROZEN = ROOT / "results" / "frozen"
REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)

In [2]:
SEED = 20260622  # repository replication seed; not claimed to be the lost historical seed
TRIALS = 12
TASKS = ["controlled_d20_white_plus_distractor", "memory_d10", "narma10", "lorenz_x"]
CONFIG = dict(N=60, K=13, lengths=(1200,500,500), washout=100, input_scale=0.8, ridge=1e-5)
hist = pd.read_csv(FROZEN / "historical_reference_metrics.csv")
display(hist[hist.notebook_id == "M19b"])

,notebook_id,metric,value,ci_low,ci_high,provenance
4,M19b,near_containment_true,0.79170,NaN,NaN,independent M19b study
5,M19b,near_containment_shuffled,0.38540,NaN,NaN,independent M19b study
6,M19b,exact_containment_true,0.63540,NaN,NaN,independent M19b study
7,M19b,exact_containment_shuffled,0.15630,NaN,NaN,independent M19b study
8,M19b,safe_minus_shuffled_gain,0.02739,0.01870,0.03710,historical manuscript result
9,M19b,inside_minus_outside_gain,0.03209,0.02342,0.04143,historical manuscript result


## Independent control replication

In [3]:
rows=[]
rng=np.random.default_rng(SEED+991)
for task in TASKS:
    for trial in range(TRIALS):
        case=tcr.evaluate_case(task,trial,"temperature",SEED,return_predictions=True,**CONFIG)
        scores=np.asarray(case["test_scores"],float); safe=np.asarray(case["safe_idxs"],int); d=int(case["default_idx"])
        K=len(scores); outside=np.setdiff1d(np.arange(K),safe)
        # Same-width contiguous intervals: exact implementation used in the preserved M23 lineage.
        controls=tcr.matched_interval_stats(scores,d,len(safe))
        # Random movement comparison is defined here transparently as one uniformly sampled alternative candidate.
        inside_candidates=safe[safe!=d]
        if len(inside_candidates)==0: inside_candidates=safe
        inside_idx=int(rng.choice(inside_candidates))
        outside_idx=int(rng.choice(outside)) if len(outside) else d
        rows.append({
            "task":task,"trial":trial,
            "near_contained":case["near_contained"],"exact_contained":case["exact_contained"],
            "matched_near_rate":controls["matched_near_rate"],"matched_oracle_rate":controls["matched_oracle_rate"],
            "safe_minus_matched_gain":case["safe_minus_matched_gain"],
            "random_inside_gain":float(scores[inside_idx]-scores[d]),
            "random_outside_gain":float(scores[outside_idx]-scores[d]),
        })
rep=pd.DataFrame(rows)
rep["inside_minus_outside_gain"]=rep.random_inside_gain-rep.random_outside_gain
rep.to_csv(REPRO/"m19b_replication_case_metrics.csv",index=False)
summary=pd.DataFrame([{
    "n_cases":len(rep),
    "true_near_containment":rep.near_contained.mean(),
    "matched_near_containment":rep.matched_near_rate.mean(),
    "true_exact_containment":rep.exact_contained.mean(),
    "matched_exact_containment":rep.matched_oracle_rate.mean(),
    "mean_safe_minus_matched_gain":rep.safe_minus_matched_gain.mean(),
    "mean_inside_minus_outside_gain":rep.inside_minus_outside_gain.mean(),
}])
summary.to_csv(REPRO/"m19b_replication_summary.csv",index=False)
display(summary.round(6))

,n_cases,true_near_containment,matched_near_containment,true_exact_containment,matched_exact_containment,mean_safe_minus_matched_gain,mean_inside_minus_outside_gain
0,48,0.770833,0.404157,0.5625,0.180598,0.029665,0.036261


The historical M19b and fresh replication use the same scientific controls, but the random-movement implementation above is documented as a reconstruction because the original run-level code was not recoverable.